# 🧬 Hydra-PaiNN: Aqueous Tautomer Ratio & Free Energy Predictor
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KeantChen/TautomerRatio/blob/main/notebooks/Hydra_PaiNN_Quickstart.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue.svg?logo=github)](https://github.com/KeantChen/TautomerRatio)

**Hydra-PaiNN** predicts aqueous tautomer equilibrium constants ($K_\text{eq}$), population percentages, and relative free energies ($\Delta G_\text{aq}$).

### 🎯 Applicability Domains:
- **Comfort Zone** ($|\Delta N_\text{aro}| = 0$, 79.4% of dataset):
  - **CBH-Taut Subset (Tier 1)**: Invariant atom hybridizations and bond types. Expected MAE: 1.29 kcal/mol.
  - **Normal Subset (Tier 2)**: Aromaticity conserved. Expected MAE: 1.97 kcal/mol.
- **Risk Zone (Tier 3)** ($|\Delta N_\text{aro}| > 0$, 20.6% of dataset):
  - Aromaticity transitions. Expected MAE: 5.14 kcal/mol (DFT reference MAE = 4.07 kcal/mol). *(Update is coming soon)*


In [ ]:
#@title 1. Environment Setup (Clone Repo & Install Dependencies)
#@markdown Run this cell to clone the repository (or update to latest commit), install dependencies, and set up xTB.
import os, sys, shutil, glob

# 1. Clone repository into Colab workspace (or update to latest commit if already cloned)
if not os.path.exists('/content/TautomerRatio'):
    print('📥 Cloning Hydra-PaiNN repository from GitHub...')
    !git clone https://github.com/KeantChen/TautomerRatio.git /content/TautomerRatio
else:
    print('🔄 Updating repository to latest commit from GitHub...')
    !cd /content/TautomerRatio && git fetch origin && git reset --hard origin/main

%cd /content/TautomerRatio

# 2. Install Python dependencies
print('📦 Installing RDKit and PyTorch Geometric...')
!pip install -q rdkit torch_geometric

# 3. Download and set up Linux xTB binary using robust extraction detection
xtb_target = '/content/TautomerRatio/bin/xtb'
if not os.path.exists(xtb_target) or os.path.getsize(xtb_target) < 1000000:
    print('⚙️ Setting up precompiled xTB 6.7.1 for Linux x86_64...')
    if not os.path.exists('/content/xtb-dist/bin/xtb'):
        !wget -q https://github.com/grimme-lab/xtb/releases/download/v6.7.1/xtb-6.7.1-linux-x86_64.tar.xz -O /content/xtb.tar.xz
        !tar -xf /content/xtb.tar.xz -C /content/
    
    os.makedirs('/content/TautomerRatio/bin', exist_ok=True)
    # Locate extracted xtb binary
    found_bins = [p for p in glob.glob('/content/**/bin/xtb', recursive=True) if not p.startswith('/content/TautomerRatio/bin')]
    if found_bins:
        src_xtb = found_bins[0]
        shutil.copy2(src_xtb, xtb_target)
        os.chmod(xtb_target, 0o755)
        try:
            shutil.copy2(src_xtb, '/usr/local/bin/xtb')
            os.chmod('/usr/local/bin/xtb', 0o755)
        except Exception:
            pass
        print(f'✅ Installed xTB binary from {src_xtb} -> {xtb_target}')
    else:
        !cp -f /content/xtb-dist/bin/xtb /content/TautomerRatio/bin/xtb
        !chmod +x /content/TautomerRatio/bin/xtb

# 4. Runtime environment configuration for xTB
share_dirs = glob.glob('/content/**/share/xtb', recursive=True)
xtb_share = share_dirs[0] if share_dirs else '/content/xtb-dist/share/xtb'

os.environ['XTBPATH'] = xtb_share
os.environ['PATH'] = f'/content/TautomerRatio/bin:/content/xtb-dist/bin:' + os.environ.get('PATH', '')
os.environ['OMP_STACKSIZE'] = '2G'
os.environ['OMP_MAX_ACTIVE_LEVELS'] = '1'

print('⚙️ Verifying xTB binary execution directly...')
!/content/TautomerRatio/bin/xtb --version

print('✅ Environment setup successfully completed! You are ready to run predictions.')


In [ ]:
#@title 2. Verify Installation & Model Loading
#@markdown Tests that PyTorch, RDKit, Scalers, Model Weights, and xTB are properly functioning.
!python verify_installation.py


In [ ]:
#@title 3. Interactive Tautomer Prediction & 2D Structure Visualization
#@markdown Enter two tautomeric SMILES representations and simulation parameters:
smiles_A = 'Oc1ccccn1'  #@param {type:'string'}
smiles_B = 'O=c1cccc[nH]1'  #@param {type:'string'}
solvent = 'water'  #@param ['water', 'gas']
temperature_K = 298.15  #@param {type:'number'}

from IPython.display import display, HTML
from rdkit import Chem
from rdkit.Chem import Draw
from tautomer_predictor import TautomerPredictor

predictor = TautomerPredictor()

print(f'🔬 Running Hydra-PaiNN prediction for:')
print(f'   Tautomer A: {smiles_A}')
print(f'   Tautomer B: {smiles_B}\n')

res = predictor.predict(smiles_A, smiles_B, temperature=temperature_K, solvent=solvent)
triage = res['triage']
pred = res['recommended_prediction']

# 2D Molecular Visualization
mol_A = Chem.MolFromSmiles(smiles_A)
mol_B = Chem.MolFromSmiles(smiles_B)
legends = [
    f"Tautomer A ({pred['population_pct']['A']:.1f}%)" + (" [Major]" if pred['major_tautomer'] == 'A' else ''),
    f"Tautomer B ({pred['population_pct']['B']:.1f}%)" + (" [Major]" if pred['major_tautomer'] == 'B' else '')
]
img = Draw.MolsToGridImage([mol_A, mol_B], molsPerRow=2, subImgSize=(320, 200), legends=legends)
display(img)

# Visual Card
tier_colors = {1: '#10b981', 2: '#3b82f6', 3: '#ef4444'}
tier_color = tier_colors.get(triage['tier'], '#6b7280')

html_card = f"""
<div style='background-color: #f8fafc; border: 1px solid #e2e8f0; border-radius: 12px; padding: 20px; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; max-width: 650px; margin-top: 15px; box-shadow: 0 4px 6px -1px rgba(0, 0, 0, 0.1);'>
  <div style='display: flex; justify-content: space-between; align-items: center; border-bottom: 2px solid #e2e8f0; padding-bottom: 12px; margin-bottom: 16px;'>
    <h3 style='margin: 0; color: #1e293b; font-size: 18px;'>Prediction Summary</h3>
    <span style='background-color: {tier_color}; color: white; padding: 4px 12px; border-radius: 9999px; font-size: 12px; font-weight: bold;'>
      Tier {triage['tier']}: {triage.get('subset', triage['tier_name'])}
    </span>
  </div>
  
  <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 16px;'>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Relative Free Energy (ΔG)</div>
      <div style='font-size: 20px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['delta_G_kcal_mol']:.2f} <span style='font-size: 13px; font-weight: normal; color: #64748b;'>kcal/mol</span></div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Equilibrium Constant (K_eq)</div>
      <div style='font-size: 20px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['K_eq']:.4e}</div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Major Tautomer</div>
      <div style='font-size: 20px; font-weight: bold; color: #2563eb; margin-top: 4px;'>Tautomer {pred['major_tautomer']}</div>
    </div>
    <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7;'>
      <div style='font-size: 11px; color: #64748b; text-transform: uppercase; font-weight: 600;'>Ratio</div>
      <div style='font-size: 18px; font-weight: bold; color: #0f172a; margin-top: 4px;'>{pred['ratio']}</div>
    </div>
  </div>

  <div style='background: white; padding: 12px; border-radius: 8px; border: 1px solid #edf2f7; font-size: 13px; color: #334155; line-height: 1.5;'>
    <strong>Domain & Subset:</strong> {triage.get('domain', 'Comfort Zone')} &rarr; {triage.get('subset', 'CBH-Taut Subset')}<br>
    <strong>Expected Benchmark MAE:</strong> ±{triage['expected_mae']:.2f} kcal/mol<br>
    <strong>Diagnostic:</strong> {triage['message']}
  </div>
</div>
"""
display(HTML(html_card))


In [ ]:
#@title 4. Batch Prediction on a CSV File
#@markdown Run predictions across multiple pairs from a CSV file (e.g. `examples/test_cases.csv` or your uploaded file).
import pandas as pd

!python predict.py --csv examples/test_cases.csv --output colab_predictions.csv

df_out = pd.read_csv('colab_predictions.csv')
print(f'✅ Batch prediction completed! Processed {len(df_out)} pairs.')
display(df_out)


In [ ]:
#@title 5. 2D Topological Triage Screening
#@markdown Screen tautomer pairs based on aromaticity conservation and CBH rules without 3D quantum calculations.
!python predict.py --csv examples/test_cases.csv --triage_only
